In [6]:
import os
import numpy as np
import copy

import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from layers import MLP

In [7]:
def sample_mask_uniform_K_per_sample(bs, d, min_K, max_K): # batch size, feature 개수, 최소 관측 샘플 수, 최대 관측 샘플 수
    m = np.zeros((bs, d), dtype=np.float32)
    Ks = np.random.randint(min_K, max_K+1, size=(bs,))
    for i, K in enumerate(Ks): # Ks의 index와 해당 index의 값
        idx = np.random.choice(d, size=K, replace=False)
        m[i, idx] = 1.0
    return m

In [8]:
def train_predictor(
    predictor,
    train_loader,
    X_val,
    y_val,
    D,
    epochs,
    optimizer,
    criterion,
    lr_factor=0.2,
    cooldown=0,
    min_lr=1e-7,
    scheduler_patience=5
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    predictor.to(device)
    X_val, y_val = X_val.to(device), y_val.to(device)

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=lr_factor,
        patience=scheduler_patience,
        cooldown=cooldown,
        min_lr=min_lr,
    )

    best_acc = 0.0
    best_state = None

    for epoch in range(epochs):
        # Train 
        predictor.train()
        total_loss = 0.0
        total_train_samples = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            m_np = sample_mask_uniform_K_per_sample(
                bs=xb.size(0),
                d=D,
                min_K=1,
                max_K=D
            )
            mb = torch.tensor(m_np, dtype=torch.float32, device=device)

            logits = predictor(xb, mb)
            loss = criterion(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)
            total_train_samples += xb.size(0)

        avg_train_loss = total_loss / total_train_samples

        # Validation 
        predictor.eval()
        with torch.no_grad():
            m_np = sample_mask_uniform_K_per_sample(
                bs=X_val.size(0),
                d=D,
                min_K=1,
                max_K=D
            )
            mv = torch.tensor(m_np, dtype=torch.float32, device=device)

            logits_val = predictor(X_val, mv)
            val_loss = criterion(logits_val, y_val).item()

            preds = logits_val.argmax(dim=-1)
            acc = (preds == y_val).float().mean().item()

        # Scheduler update
        scheduler.step(acc)

        # 로그 출력
        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"train_loss={avg_train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={acc:.4f} | "
            f"lr={current_lr:.6f}"
        )

        # Best model 저장
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(predictor.state_dict())

    # 모든 epoch 후 best 모델 복원
    if best_state is not None:
        predictor.load_state_dict(best_state)

    print(f"Best validation accuracy = {best_acc:.4f}")

In [9]:
ROOT_DIR = os.getcwd()
DATA_DIR = os.path.join(ROOT_DIR, "data", "cube")

X_train = torch.load(f"{DATA_DIR}/X_train_cdf.pt").float()
y_train = torch.load(f"{DATA_DIR}/y_train.pt").long()

X_val   = torch.load(f"{DATA_DIR}/X_val_cdf.pt").float()
y_val   = torch.load(f"{DATA_DIR}/y_val.pt").long()

X_test = torch.load(f"{DATA_DIR}/X_test_cdf.pt").float()
y_test = torch.load(f"{DATA_DIR}/y_test.pt").long()

In [10]:
batch_size = 128
epochs = 100
lr = 0.0003
weight_decay = 1e-4
D = 20  # feature 개수 고정

predictor = MLP(in_dim=20, hidden_dim=250, out_dim=8, num_hidden=2)
optimizer = AdamW(predictor.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()

train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)

train_predictor(
    predictor=predictor,
    train_loader=train_loader,
    X_val=X_val,
    y_val=y_val,
    D=D,
    epochs=epochs,
    optimizer=optimizer,
    criterion=criterion
)

Epoch 1/100 | train_loss=0.3361 | val_loss=0.1864 | val_acc=0.9370 | lr=0.000300
Epoch 2/100 | train_loss=0.1761 | val_loss=0.1458 | val_acc=0.9468 | lr=0.000300
Epoch 3/100 | train_loss=0.1416 | val_loss=0.1250 | val_acc=0.9506 | lr=0.000300
Epoch 4/100 | train_loss=0.1224 | val_loss=0.1152 | val_acc=0.9535 | lr=0.000300
Epoch 5/100 | train_loss=0.1109 | val_loss=0.1060 | val_acc=0.9560 | lr=0.000300
Epoch 6/100 | train_loss=0.1039 | val_loss=0.1021 | val_acc=0.9589 | lr=0.000300
Epoch 7/100 | train_loss=0.0981 | val_loss=0.0986 | val_acc=0.9586 | lr=0.000300
Epoch 8/100 | train_loss=0.0928 | val_loss=0.0997 | val_acc=0.9589 | lr=0.000300
Epoch 9/100 | train_loss=0.0890 | val_loss=0.0922 | val_acc=0.9611 | lr=0.000300
Epoch 10/100 | train_loss=0.0849 | val_loss=0.0907 | val_acc=0.9634 | lr=0.000300
Epoch 11/100 | train_loss=0.0831 | val_loss=0.0921 | val_acc=0.9626 | lr=0.000300
Epoch 12/100 | train_loss=0.0804 | val_loss=0.0945 | val_acc=0.9624 | lr=0.000300
Epoch 13/100 | train_loss

KeyboardInterrupt: 